In [31]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi
import skimage as ski
from PIL import Image
import cv2
from skimage.filters import sato

In [32]:
IMAGE_ID = 'S1616-2_a'

In [33]:
base_path = f"../data/{IMAGE_ID}/"
orig_img = cv2.imread(f"{base_path}/image.png")[:, :, 1]
mask_img = cv2.imread(f"{base_path}/mask.png", cv2.IMREAD_GRAYSCALE)
annotation_img = cv2.imread(f"{base_path}/annotation.png", cv2.IMREAD_GRAYSCALE)
label_img = cv2.imread(f"{base_path}/label.png", cv2.IMREAD_GRAYSCALE)

In [34]:
print(f"Original Image shape: {orig_img.shape}, dtype: {orig_img.dtype}")
print(f"Mask Image shape: {mask_img.shape}, dtype: {mask_img.dtype}")
print(f"Annotation Image shape: {annotation_img.shape}, dtype: {annotation_img.dtype}")
print(f"Label Image shape: {label_img.shape}, dtype: {label_img.dtype}")

Original Image shape: (1024, 3789), dtype: uint8
Mask Image shape: (1024, 3789), dtype: uint8
Annotation Image shape: (1024, 3789), dtype: uint8
Label Image shape: (2048, 7577), dtype: uint8


In [29]:
extended_label = cv2.dilate(label_img, np.ones((5, 5), dtype=np.uint8), iterations=1)

In [30]:
components = ski.measure.label(annotation_img, connectivity=2)
valid_annotations =[]
invalid_annotations = []
for region in ski.measure.regionprops(components):
    minr, minc, maxr, maxc = region.bbox
    region_mask = (components[minr:maxr, minc:maxc] == region.label)
    annotation_region = extended_label[minr:maxr, minc:maxc]
    if np.any((annotation_region == 255) & region_mask):
        valid_annotations.append(region.label)
    else:
        invalid_annotations.append(region.label)

viz_img = cv2. cvtColor(orig_img.copy(), cv2.COLOR_GRAY2BGR)
# viz_img = cv2.addWeighted(viz_img, 0.7, cv2.cvtColor(label_img, cv2.COLOR_GRAY2BGR), 0.3, 0)
viz_img[annotation_img == 255] = [255, 0, 0]
for invalid_annotation in invalid_annotations:
    viz_img[components == invalid_annotation] = [0, 0, 255]

cv2.imwrite(f"{base_path}/invalid_annotation_viz.png", viz_img)
# plt.figure(figsize=(256, 200))
# plt.subplot(2, 2, 1)
# plt.title("Original Image")
# plt.imshow(viz_img)
# plt.axis('off')

True